# Generate PDF Report
Run this cell after completing all analysis cells. It reads the saved charts from `reports/` and produces a 7-page professional PDF.

In [ ]:
# Install reportlab if needed
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'reportlab', '-q'])
print('reportlab ready')

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image,
    Table, TableStyle, HRFlowable, PageBreak
)
import os

W, H   = A4
REPORTS = '../reports'
OUTPUT  = os.path.join(REPORTS, 'ecommerce_analysis_report.pdf')

print('Imports OK')

In [ ]:
# ── Colours
NAVY   = colors.HexColor('#1a2a4a')
BLUE   = colors.HexColor('#4C72B0')
LIGHT  = colors.HexColor('#f0f4fb')
GREY   = colors.HexColor('#666666')
GREEN  = colors.HexColor('#2ecc71')
RED    = colors.HexColor('#e74c3c')
ORANGE = colors.HexColor('#e67e22')
WHITE  = colors.white

# ── Styles
def S(name, **kw):
    return ParagraphStyle(name, **kw)

TITLE   = S('rTitle',   fontSize=26, textColor=WHITE,  fontName='Helvetica-Bold',    leading=32, spaceAfter=4)
SUB     = S('rSub',     fontSize=12, textColor=LIGHT,  fontName='Helvetica',         leading=16)
H1      = S('rH1',      fontSize=16, textColor=NAVY,   fontName='Helvetica-Bold',    leading=20, spaceBefore=14, spaceAfter=6)
H2      = S('rH2',      fontSize=12, textColor=BLUE,   fontName='Helvetica-Bold',    leading=16, spaceBefore=10, spaceAfter=4)
BODY    = S('rBody',    fontSize=10, textColor=colors.HexColor('#333333'),
            fontName='Helvetica', leading=15, spaceAfter=4)
CAPTION = S('rCaption', fontSize=8,  textColor=GREY,   fontName='Helvetica-Oblique', leading=11, spaceAfter=8)
INSIGHT = S('rInsight', fontSize=10, textColor=NAVY,   fontName='Helvetica',
            leading=14, leftIndent=12, spaceAfter=3)

print('Styles OK')

In [ ]:
# ── Helpers
def rule():
    return HRFlowable(width='100%', thickness=0.5, color=colors.HexColor('#dddddd'), spaceAfter=8)

def chart(filename, caption, width=15*cm):
    path = os.path.join(REPORTS, filename)
    if not os.path.exists(path):
        return [Paragraph(f'[Chart not found: {filename}]', CAPTION)]
    img = Image(path, width=width, height=width*0.42)
    img.hAlign = 'CENTER'
    cap = Paragraph(caption, CAPTION)
    cap.hAlign = 'CENTER'
    return [img, cap]

def kpi_table(rows):
    data = [[Paragraph(k, S('k', fontSize=9, fontName='Helvetica-Bold', textColor=GREY)),
             Paragraph(v, S('v', fontSize=10, fontName='Helvetica-Bold', textColor=NAVY))]
            for k, v in rows]
    t = Table(data, colWidths=[6*cm, 9*cm])
    t.setStyle(TableStyle([
        ('BACKGROUND',    (0,0), (-1,-1), LIGHT),
        ('ROWBACKGROUNDS',(0,0), (-1,-1), [LIGHT, WHITE]),
        ('GRID',          (0,0), (-1,-1), 0.3, colors.HexColor('#cccccc')),
        ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING',    (0,0), (-1,-1), 6),
        ('BOTTOMPADDING', (0,0), (-1,-1), 6),
        ('LEFTPADDING',   (0,0), (-1,-1), 10),
    ]))
    return t

def seg_table():
    header = ['Segment', 'Customers', 'Revenue (£)', '% of Total', 'Avg Spend']
    rows = [
        ['Champions',       '799',   '4,911,049', '55.1%', '£6,146'],
        ['Loyal Customers', '1,130', '2,100,164', '23.6%', '£1,859'],
        ['At Risk',         '1,066', '1,100,861', '12.4%', '£1,033'],
        ['Lost',            '1,084',   '707,218',  '7.9%',   '£652'],
        ['Promising',         '185',    '67,388',  '0.8%',   '£364'],
        ['New Customers',      '74',    '24,728',  '0.3%',   '£334'],
    ]
    seg_colors = [GREEN, BLUE, ORANGE, RED,
                  colors.HexColor('#f39c12'), colors.HexColor('#9b59b6')]
    tdata = [[Paragraph(h, S('th', fontSize=9, fontName='Helvetica-Bold', textColor=WHITE))
              for h in header]]
    for row in rows:
        tdata.append([Paragraph(c, S('td', fontSize=9, fontName='Helvetica',
                                     textColor=colors.HexColor('#222222'))) for c in row])
    t = Table(tdata, colWidths=[4.5*cm, 2.5*cm, 3*cm, 2.5*cm, 2.5*cm])
    sl = [
        ('BACKGROUND',    (0,0), (-1,0), NAVY),
        ('GRID',          (0,0), (-1,-1), 0.3, colors.HexColor('#cccccc')),
        ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING',    (0,0), (-1,-1), 6),
        ('BOTTOMPADDING', (0,0), (-1,-1), 6),
        ('LEFTPADDING',   (0,0), (-1,-1), 8),
        ('ROWBACKGROUNDS',(0,1), (-1,-1), [LIGHT, WHITE]),
    ]
    for i, c in enumerate(seg_colors):
        sl.append(('TEXTCOLOR', (0,i+1), (0,i+1), c))
        sl.append(('FONTNAME',  (0,i+1), (0,i+1), 'Helvetica-Bold'))
    t.setStyle(TableStyle(sl))
    return t

print('Helpers OK')

In [ ]:
# ── Page callbacks
def cover_page(canvas, doc):
    canvas.saveState()
    canvas.setFillColor(NAVY)
    canvas.rect(0, 0, W, H, fill=1, stroke=0)
    canvas.setFillColor(BLUE)
    canvas.rect(0, H*0.55, W, H*0.45, fill=1, stroke=0)
    canvas.setFillColor(colors.HexColor('#2a3f6a'))
    canvas.rect(0, H*0.52, W, H*0.05, fill=1, stroke=0)
    canvas.restoreState()

def later_pages(canvas, doc):
    canvas.saveState()
    canvas.setFillColor(NAVY)
    canvas.rect(0, 0, W, 1.2*cm, fill=1, stroke=0)
    canvas.setFillColor(WHITE)
    canvas.setFont('Helvetica', 8)
    canvas.drawString(2*cm, 0.45*cm, 'E-Commerce Sales Analysis  |  Mahdi Almasi')
    canvas.drawRightString(W - 2*cm, 0.45*cm, f'Page {doc.page}')
    canvas.restoreState()

print('Page callbacks OK')

In [ ]:
# ── Build story
doc   = SimpleDocTemplate(OUTPUT, pagesize=A4,
                          leftMargin=2*cm, rightMargin=2*cm,
                          topMargin=2.5*cm, bottomMargin=2*cm)
story = []

# Page 1 — Cover
story.append(Spacer(1, 5.5*cm))
story.append(Paragraph('E-Commerce Sales Analysis', TITLE))
story.append(Paragraph('UK Online Retail Dataset  |  Dec 2010 – Dec 2011', SUB))
story.append(Spacer(1, 0.5*cm))
story.append(Paragraph('Prepared by Mahdi Almasi  ·  Data Analyst & ML Engineer', SUB))
story.append(PageBreak())

# Page 2 — Overview
story.append(Paragraph('1. Project Overview', H1))
story.append(rule())
story.append(Paragraph(
    'This report presents a complete end-to-end analysis of a UK-based online retailer '
    'operating between December 2010 and December 2011. The analysis covers data cleaning, '
    'exploratory analysis, customer segmentation using the RFM framework, and a predictive '
    'model for customer revenue forecasting.', BODY))
story.append(Spacer(1, 0.3*cm))
story.append(Paragraph('Dataset Summary', H2))
story.append(kpi_table([
    ('Raw records',      '541,909 transactions'),
    ('After cleaning',   '397,884 transactions  (144,025 removed)'),
    ('Date range',       '01 Dec 2010 → 09 Dec 2011'),
    ('Total revenue',    '£8,911,407.90'),
    ('Unique customers', '4,338'),
    ('Unique products',  '3,665'),
    ('Countries served', '37'),
]))
story.append(Spacer(1, 0.3*cm))
story.append(Paragraph('Data Cleaning Steps', H2))
for step in [
    'Removed 135,080 rows with missing CustomerID (required for RFM)',
    'Removed cancelled orders (InvoiceNo starting with C)',
    'Removed rows with Quantity <= 0 or UnitPrice <= 0',
    'Derived Revenue = Quantity x UnitPrice',
    'Parsed InvoiceDate and extracted Month and DayOfWeek features',
]:
    story.append(Paragraph(f'• {step}', INSIGHT))
story.append(PageBreak())

# Page 3 — Revenue
story.append(Paragraph('2. Revenue Analysis', H1))
story.append(rule())
story.append(Paragraph('Monthly Revenue Trend', H2))
story.append(Paragraph(
    'Revenue was relatively stable in early 2011 (~£580k/month), then surged from September '
    'onwards as the holiday season approached. November 2011 peaked at £1.16M. The December '
    '2011 drop reflects incomplete data (dataset ends 9 Dec).', BODY))
story.extend(chart('01_monthly_revenue.png', 'Fig 1. Monthly revenue from Dec 2010 to Dec 2011. Peak in Nov 2011.'))
story.append(Spacer(1, 0.3*cm))
story.append(Paragraph('Revenue by Day of Week', H2))
story.append(Paragraph(
    'Thursday drives the highest revenue (£1.97M), while Saturday records £0 — '
    'the business does not operate on Saturdays. Sunday generates only £797k, '
    'suggesting limited weekend trading.', BODY))
story.extend(chart('04_day_of_week.png', 'Fig 2. Revenue by day of week. Saturday shows zero activity.'))
story.append(PageBreak())

# Page 4 — Products & Countries
story.append(Paragraph('3. Products & Markets', H1))
story.append(rule())
story.append(Paragraph('Top 10 Products by Revenue', H2))
story.append(Paragraph(
    "'Paper Craft, Little Birdie' leads at £168,470, followed by 'Regency Cakestand 3 Tier' "
    'at £142,593. The presence of Postage and Manual in the top 10 suggests these '
    'are administrative line items that may warrant exclusion in future analyses.', BODY))
story.extend(chart('02_top_products.png', 'Fig 3. Top 10 products by total revenue.'))
story.append(Spacer(1, 0.3*cm))
story.append(Paragraph('Top International Markets', H2))
story.append(Paragraph(
    'Excluding the UK (which dominates at ~85% of volume), Netherlands leads at £285,446, '
    'followed by EIRE (Ireland) at £265,546 and Germany at £228,867. The strong EIRE result '
    'likely reflects geographic and cultural proximity to the UK.', BODY))
story.extend(chart('03_top_countries.png', 'Fig 4. Top 10 countries by revenue, excluding United Kingdom.'))
story.append(PageBreak())

# Page 5 — RFM
story.append(Paragraph('4. Customer Segmentation (RFM)', H1))
story.append(rule())
story.append(Paragraph(
    'RFM (Recency, Frequency, Monetary) analysis assigns each customer a score across '
    'three dimensions. Customers were then grouped into 6 actionable segments based '
    'on their score combination.', BODY))
story.append(Spacer(1, 0.2*cm))
story.append(seg_table())
story.append(Spacer(1, 0.4*cm))
story.extend(chart('05_rfm_segments.png', 'Fig 5. Customer count and average revenue per RFM segment.'))
story.append(Spacer(1, 0.3*cm))
story.append(Paragraph('Key Insight', H2))
story.append(Paragraph(
    'Champions (799 customers, 18.4% of base) generate £4.91M — 55.1% of total revenue. '
    'They spend 9.4x more on average than Lost customers. Meanwhile, 2,150 At Risk and '
    'Lost customers represent £1.8M in revenue at risk of permanent churn.', BODY))
story.append(PageBreak())

# Page 6 — Model
story.append(Paragraph('5. Predictive Model', H1))
story.append(rule())
story.append(Paragraph(
    'A Random Forest Regressor was trained to predict customer revenue from RFM features. '
    'Revenue was log-transformed to handle the extreme right skew caused by high-value '
    'wholesale customers (max spend: £280,206).', BODY))
story.append(Paragraph('Model Performance', H2))
story.append(kpi_table([
    ('Algorithm',          'Random Forest Regressor (100 trees)'),
    ('Target',             'Customer lifetime revenue (log-transformed)'),
    ('Train / Test split', '80% / 20%'),
    ('R² score',           '0.597  (model explains 59.7% of revenue variance)'),
    ('MAE',                '£512.53  (average prediction error)'),
    ('Top feature',        'M_Score (past monetary behaviour predicts future spend)'),
]))
story.append(Spacer(1, 0.3*cm))
story.extend(chart('06_model_performance.png', 'Fig 6. Actual vs predicted revenue (left) and feature importance (right).'))
story.append(Spacer(1, 0.2*cm))
story.append(Paragraph(
    'Note: Outliers above the 99th percentile were excluded from model training '
    'to prevent a small number of wholesale buyers from distorting predictions.', BODY))
story.append(PageBreak())

# Page 7 — Recommendations
story.append(Paragraph('6. Recommendations', H1))
story.append(rule())
recs = [
    ('Retain Champions',
     '799 customers generate 55% of revenue. Introduce a VIP loyalty programme, '
     'early access to new products, and personalised offers to reduce churn risk.'),
    ('Re-engage At Risk customers',
     '1,066 customers have gone quiet but previously spent an average of £1,033. '
     'A targeted win-back email campaign with a discount incentive could recover '
     'a significant portion of this £1.1M segment.'),
    ('Expand Netherlands & EIRE',
     'Both markets outperform Germany and France despite likely smaller marketing spend. '
     'Investigate whether localised campaigns or dedicated account management '
     'would accelerate growth in these high-potential markets.'),
    ('Investigate Saturday closure',
     'Zero revenue on Saturdays is an anomaly worth examining. Even partial Saturday '
     'operations or automated online order processing could unlock additional revenue.'),
    ('Clean product catalogue',
     "'Postage' and 'Manual' appearing in the top 10 revenue items suggests the "
     'product catalogue needs cleaning to separate physical goods from admin charges.'),
]
for i, (title, body) in enumerate(recs, 1):
    story.append(Paragraph(f'{i}.  {title}', H2))
    story.append(Paragraph(body, BODY))
    story.append(Spacer(1, 0.1*cm))

story.append(Spacer(1, 0.5*cm))
story.append(rule())
story.append(Paragraph(
    'Analysis performed using Python (pandas, matplotlib, seaborn, scikit-learn). '
    'Dataset: UCI Online Retail Dataset (Chen et al., 2012). '
    'GitHub: github.com/yourusername/ecommerce-analysis', CAPTION))

# ── Render
doc.build(story, onFirstPage=cover_page, onLaterPages=later_pages)
print(f'PDF saved → {OUTPUT}')